In [1]:
!pip install huggingface_hub[hf_xet] pandas transformers torch

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import re
import pandas as pd

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("njlr/cs180-project")
model = AutoModelForSequenceClassification.from_pretrained("njlr/cs180-project")
model.eval()

# Preprocoess
def clean_text(s: str):
    # Replace all curly double quotes with "
    s = re.sub(r'[“”]', '"', s)
     
    # Replace all curly single quotes with '
    s = re.sub(r"[‘’]", "'", s)
    
    # Replace en dash and em dash with hyphen
    s = re.sub(r"[–—]", "-", s)

    # Only retain alphanumeric, whitespace characters, single and double quotes, and hyphens
    s = re.sub(pattern=rf"[^a-zA-Z0-9\s\-\'\"]", repl="", string=s, flags=re.IGNORECASE)

    # Remove extra whitespaces
    s = re.sub(pattern=r"\s+", repl=" ", string=s).strip()

    return s

def preprocess(text: str):
    return clean_text(text)

df_test = pd.read_csv('../data/demo.csv')
df_test['cleaned'] = df_test['text'].apply(preprocess)

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    model.config.id2label = {0: "Risk", 1: "Neutral", 2: "Opportunity"}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)
        confidence, predicted_class = torch.max(probs, dim=1)
        label = model.config.id2label[predicted_class.item()]
    return label, confidence.item()

df_test[['predicted_label', 'confidence']] = df_test['cleaned'].apply(
    lambda x: pd.Series(predict(x))
)

# Apply prediction on each line of text.
for _, row in df_test.iterrows():
    print(f"Text: {row['text']}\nPredicted Label: {row['predicted_label']}\nConfidence: {row['confidence']}\n")

c:\Users\Lenovo\Documents\GitHub\cs180-nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Text: With reference to the reuse of materials, the projects created in implementation of the Group’s investment plans for the Italian motorway network provided for the reuse - within the regulatory limits - of the earth deriving from excavations, in order to mitigate the environmental impact linked mainly to the procurement of inert quarry materials and the disposal in landfills of unused materials. They are reused to create embankments, landscaping and noise-absorbing dunes, as well as for the redevelopment of degraded areas (such as abandoned quarries).
Predicted Label: Neutral
Confidence: 0.9024878144264221

Text: 2019 also saw the continuation of activities regarding lighting, with widespread use of LED technology, both in motorway tunnels and airports, as well as in buildings, which reduced electrical energy consumption by around 5.4 GWh. As regards air conditioning, modernisation of the systems continued, with more efficient machinery, such as refrigeration units, significant re